In [ ]:
import os
import gc
import numpy as np
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
import imageio.v3 as iio
import pandas as pd
from typing import Optional

def read_tif_raw(path: Path) -> np.ndarray:
    """Robust TIFF reader → float32 2D array."""
    arr = iio.imread(path)
    arr = np.asarray(arr)

    if arr.ndim == 3:
        if arr.shape[-1] in (3, 4):
            arr = arr[..., :3].mean(axis=-1)  # RGB(A) → grayscale
        else:
            arr = arr[0]  # first frame of stack

    return arr.astype(np.float32, copy=False)


def read_mask(path: Path) -> np.ndarray:
    """Read mask image; return boolean array (True = keep, False = masked)."""
    arr = iio.imread(path)
    arr = np.asarray(arr)
    if arr.ndim > 2:
        arr = arr[..., 0]
    mask = arr != 0
    return mask


def resolve_mask_path(mask_path_raw) -> Optional[Path]:
    """Return a valid mask file path, or None."""
    if mask_path_raw is None:
        return None

    mask_path = Path(mask_path_raw)
    if mask_path.is_file():
        return mask_path

    if mask_path.is_dir():
        candidates = sorted(
            p for p in mask_path.iterdir()
            if p.suffix.lower() in (".tif", ".tiff", ".png")
        )
        if candidates:
            return candidates[0]
        else:
            print(f"No image files found in mask folder: {mask_path}")
            return None

    print(f"Mask path does not exist: {mask_path}")
    return None

def qx_qy_map(
    shape,
    cx_pix,
    cy_pix,
    pix_um: float,
    L_mm: float,
    lam_A: float,
):
    """Compute qx, qy for each detector pixel."""
    nrows, ncols = shape
    k = 2.0 * np.pi / lam_A
    px_mm = pix_um * 1e-3

    yy, xx = np.indices((nrows, ncols), dtype=np.float64)

    x_mm = (xx - cx_pix) * px_mm
    y_mm = (yy - cy_pix) * px_mm

    r = np.sqrt(x_mm**2 + y_mm**2 + L_mm**2)

    qx = k * x_mm / r
    qy = k * y_mm / r
    return qx, qy


def save_q_vs_I_plot(qx_line, I_line, out_png: Path, sample_name: str,
                     log_y: bool = True):
    """Save 1D qx vs I plot."""
    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)

    fig = plt.figure(figsize=(6, 4))
    plt.plot(qx_line, I_line, marker="", linestyle="-")

    if log_y:
        plt.yscale("log")

    plt.xlabel("qx ($\mathrm{{\AA}}^{{-1}}$)")
    plt.ylabel("Intensity (au)")
    plt.title(sample_name[:20])  # first 20 characters
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    
    # Explicit cleanup
    plt.clf()
    plt.close(fig)
    print(f"Saved 1D qx vs I plot: {out_png}")


def save_2d_with_linecut(
    img,
    qx,
    qy,
    row_idx: int,
    band_halfwidth: int,
    out_png: Path,
    q_range_x=None,
    q_range_y=None,
    labels: bool = True,
    show: bool = False,
):
    """
    Save 2D qx–qy map with a horizontal stripe showing the
    band of pixels used for the line cut, plus the central line.
    """
    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)

    qy_plot = -qy
    qx_edges = 0.5 * (qx[:-1, :-1] + qx[:-1, 1:])
    qy_edges = 0.5 * (qy_plot[:-1, :-1] + qy_plot[1:, :-1])

    finite = img[np.isfinite(img) & (img > 0)]
    if finite.size > 0:
        vmin = float(np.percentile(finite, 1))
        vmax = float(np.percentile(finite, 99))
        if not (np.isfinite(vmin) and np.isfinite(vmax) and vmin < vmax):
            vmin, vmax = 1e-6, float(np.nanmax(finite)) or 1.0
    else:
        vmin, vmax = 1e-6, 1.0
    img_clipped = np.clip(img, vmin, vmax)

    fig = plt.figure(figsize=(6, 5))
    mesh = plt.pcolormesh(
        qx_edges,
        qy_edges,
        img_clipped[:-1, :-1],
        norm=LogNorm(vmin=vmin, vmax=vmax),
        shading="auto",
    )

    top_row = max(0, row_idx - band_halfwidth)
    bot_row = min(qy_plot.shape[0] - 1, row_idx + band_halfwidth)

    qy_top = np.nanmedian(qy_plot[top_row, :])
    qy_bot = np.nanmedian(qy_plot[bot_row, :])
    y_min, y_max = min(qy_top, qy_bot), max(qy_top, qy_bot)

    plt.axhspan(y_min, y_max, facecolor="white", alpha=0.15, edgecolor="red", linewidth=1.0)
    plt.plot(qx[row_idx, :], qy_plot[row_idx, :], color="white", linewidth=1.0)

    if q_range_x is not None: plt.xlim(q_range_x)
    if q_range_y is not None: plt.ylim(q_range_y)

    if labels:
        plt.colorbar(mesh, label="Log(Intensity)")
        plt.xlabel("qx ($\mathrm{{\AA}}^{{-1}}$)")
        plt.ylabel("qy ($\mathrm{{\AA}}^{{-1}}$)")
        plt.title("2D q map with line-cut band")
    else:
        plt.axis("off")

    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    print(f"Saved 2D map with line cut band: {out_png}")

    if show:
        plt.show()

    plt.clf()
    plt.close(fig)


def process_tif_folder(cfg):
    base = Path(cfg.results_base)
    plots_dir = base / "q_vs_I_plots"
    linecut_dir = base / "2D_with_Line_Cut"
    base.mkdir(parents=True, exist_ok=True)

    folder = Path(cfg.input_folder)
    files = sorted([f for f in folder.iterdir() if f.suffix.lower() in (".tif", ".tiff")])

    if not files:
        print("  No TIFF files found.")
        return

    mask = None
    if cfg.mask_path is not None:
        mask_file = resolve_mask_path(cfg.mask_path)
        if mask_file:
            mask = read_mask(mask_file)
            print(f"Loaded mask: {mask_file}")

    print(f"\nFound {len(files)} TIFF files.")
    
    per_sample = {}
    sample_counters = {}

    for i, f in enumerate(files):
        print(f"[{i+1}/{len(files)}] {f.name}")

        img = read_tif_raw(f)
        img_masked = np.where(mask, img, np.nan) if (mask is not None and mask.shape == img.shape) else img

        qx, qy = qx_qy_map(
            img_masked.shape, cfg.cx_pix, cfg.cy_pix, cfg.pix_um, cfg.L_mm, cfg.lam_A
        )

        row_idx = int(round(cfg.cy_pix - cfg.line_offset_pix))
        row_idx = max(0, min(row_idx, img_masked.shape[0] - 1))
        
        top_row = max(0, row_idx - cfg.line_band_halfwidth)
        bot_row = min(img_masked.shape[0] - 1, row_idx + cfg.line_band_halfwidth)

        I_band = np.nanmean(img_masked[top_row:bot_row + 1, :], axis=0)
        row_qx = qx[row_idx, :]

        side_mask = row_qx > 0
        qx_line, I_line = row_qx[side_mask], I_band[side_mask]

        valid = np.isfinite(I_line) & np.isfinite(qx_line) & (I_line > 0)
        qx_line, I_line = qx_line[valid], I_line[valid]

        sample_name = f.stem
        save_q_vs_I_plot(qx_line, I_line, plots_dir / f"{sample_name}_q_vs_I.png", sample_name, cfg.use_log_intensity)
        
        save_2d_with_linecut(
            img_masked, qx, qy, row_idx, cfg.line_band_halfwidth,
            linecut_dir / f"{sample_name}_2D_with_linecut.png",
            cfg.q_range_x, cfg.q_range_y, cfg.show_labels, show=(i == 0)
        )

        # Build column key
        sample_key_base = sample_name[:20]
        count = sample_counters.get(sample_key_base, 0)
        sample_key = sample_key_base if count == 0 else f"{sample_key_base}_{count}"
        sample_counters[sample_key_base] = count + 1
        per_sample[sample_key] = (qx_line.copy(), I_line.copy())

        # CRITICAL: Clean up memory every 10 files
        if i % 10 == 0:
            plt.close('all')
            gc.collect()

    # Save Excel
    if per_sample:
        max_len = max(len(v[0]) for v in per_sample.values())
        out_dict = {}
        for key, (q_arr, I_arr) in per_sample.items():
            q_pad, I_pad = np.full(max_len, np.nan), np.full(max_len, np.nan)
            q_pad[:len(q_arr)], I_pad[:len(I_arr)] = q_arr, I_arr
            out_dict[f"q_{key}"], out_dict[f"I_{key}"] = q_pad, I_pad

        pd.DataFrame(out_dict).to_excel(base / "q_vs_I_data.xlsx", index=False)
        print("\nSaved combined Excel data.")

In [ ]:
class Config:
    # Input TIFFs
    input_folder = "../blends_tiff/"

    # Mask location:
    mask_path = "./masks/M_mask.png"

    # Output base folder
    results_base = "./outputs/"

    # Geometry
    L_mm   = 5030.0      # sample-to-detector distance (mm)
    lam_A  = 0.9184      # wavelength (Angstrom)
    pix_um = 172.0       # pixel size (microns)
    cx_pix = 754         # beam center x (pixels)
    cy_pix = 1100        # beam center y (pixels)
    # cy_pix = 1085        # beam center y (pixels)

    # Line-cut settings
    line_offset_pix = 135     # pixels above beam center (positive = up)
    line_band_halfwidth = 10   # stripe half-width in pixels (2 → 5-pixel band)
    use_log_intensity = True  # log-scale y-axis in 1D plot

    # q-range for 2D display (optional)
    q_range_x = (-0.06, 0.06)
    q_range_y = (-0.04, 0.10)

    # Show labels on plots
    show_labels = True

if __name__ == "__main__":
    process_tif_folder(Config())